# 用最少的枚数付款 ― 用 Optimize 做最优化

## 本章要点

Z3 要回答的问题有两类。

- **约束满足问题**：找出一个满足条件的解
- **最优化问题**：在满足条件的解中，找最好的一个

本章用两种方法解同一个“硬币枚数”的问题，看看差别在哪里。

- 用 `Solver` 收集解再比较（只能一枚一枚地数）
- 用 `Optimize` 直接求“枚数最少的解”
- 用 `lower()` / `upper()` 确认“这确实是最优”

## 问题

要付 134 美分，怎样组合才能用尽量少的硬币？

- 可用的硬币：1, 2, 5, 10, 25, 50 美分
- 要凑的金额：134 美分
- 目标：**让使用的硬币枚数尽量少**

“让合计等于 134 美分”是约束，“让枚数最小”是目标。在 Z3 里，这两件事要分开写。这就是本章的要点。

## 先用 Solver 收集所有满足条件的解

`Solver` 是“找出满足条件的解”的工具。我们想知道最少的枚数，所以这次把所有满足条件的解都取出来，自己比较枚数。

In [1]:
from z3 import *
from helper.z3 import all_solutions   # 本书准备的“依次返回全部解”函数

coins = [1, 2, 5, 10, 25, 50]   # 可用的硬币种类（美分）
total = 134                     # 要凑的金额
max_coins = 8                   # 先限定在“8 枚以下”

counts = [Int(f"n_{c}") for c in coins]                # 每种硬币的枚数
amount = Sum([c * n for c, n in zip(coins, counts)])   # 合计金额
num_coins = Sum(counts)                               # 合计枚数

solver = Solver()
for n in counts:
    solver.add(n >= 0)              # 枚数在 0 以上
solver.add(amount == total)         # 合计正好 134 美分
solver.add(num_coins <= max_coins)  # 枚数在 8 枚以下


def show(model):
    """把解显示成“8 枚: 2 美分×2, 5 美分×1, ...”的形式"""
    used = {c: model.eval(n).as_long() for c, n in zip(coins, counts)}
    used = {c: k for c, k in used.items() if k > 0}   # 省略 0 枚的硬币
    print(f"{model.eval(num_coins)} 枚: " + ", ".join(f"{c} 美分×{k}" for c, k in used.items()))


for model in all_solutions(solver):
    show(model)

8 枚: 2 美分×2, 5 美分×1, 25 美分×5
8 枚: 1 美分×2, 2 美分×1, 10 美分×3, 50 美分×2
8 枚: 1 美分×1, 2 美分×4, 25 美分×1, 50 美分×2
8 枚: 1 美分×4, 5 美分×1, 25 美分×1, 50 美分×2
7 枚: 2 美分×2, 5 美分×1, 25 美分×3, 50 美分×1
8 枚: 2 美分×2, 10 美分×3, 25 美分×2, 50 美分×1
8 枚: 1 美分×2, 2 美分×1, 5 美分×1, 25 美分×3, 50 美分×1
7 枚: 2 美分×2, 10 美分×3, 50 美分×2
8 枚: 2 美分×2, 5 美分×2, 10 美分×2, 50 美分×2
7 枚: 1 美分×2, 2 美分×1, 5 美分×1, 25 美分×1, 50 美分×2
6 枚: 2 美分×2, 5 美分×1, 25 美分×1, 50 美分×2


满足条件的解找到了 **11 种**。最少的是 **6 枚** 的组合。

不过这种方法有弱点。

- `Solver` 并不是按“枚数从少到多”的顺序搜索的。这次它最后才出现，但这个顺序没有保证。
- 因为要把所有解收集起来比较，解一多就很花时间。实际上，去掉 `max_coins` 的上限后，解有 **11,713 种**。
- 上限 8 枚也是我们凭感觉定的数字。并没有证明“没有比 6 枚更少的付法”。

为了“找最好的解”而准备的工具，就是 `Optimize`。

## 用 Optimize 直接求“枚数最少的解”

`Optimize` 在满足条件的解中，寻找让**目标函数**（这里是枚数）最小或最大的解。用法和 `Solver` 几乎一样，只是多了一句 `minimize()`。

In [2]:
opt = Optimize()

for n in counts:
    opt.add(n >= 0)
opt.add(amount == total)        # 金额条件相同。枚数上限已经不需要了

obj = opt.minimize(num_coins)   # 目标函数：让合计枚数最小

print(opt.check())

model = opt.model()
show(model)
print("合计:", sum(c * model.eval(n).as_long() for c, n in zip(coins, counts)), "美分")

sat
6 枚: 2 美分×2, 5 美分×1, 25 美分×1, 50 美分×2
合计: 134 美分


答案还是同样的 6 枚。`minimize()` 会返回“目标函数的句柄”。用这个句柄，就能确认最优值到底能确定到什么程度。

In [3]:
print("最优值的下界:", opt.lower(obj))
print("最优值的上界:", opt.upper(obj))

# 把条件改差一点比较看看（加上 7 枚以上，就选不到 6 枚的解）
opt.add(num_coins >= 7)
opt.check()
print("在 7 枚以上的条件下:", opt.lower(obj), opt.upper(obj))

最优值的下界: 6
最优值的上界: 6
在 7 枚以上的条件下: 7 7


`lower()` 和 `upper()` 是“最优值就在这个范围内”的保证。第一个结果里两者都是 6，也就是说 Z3 表明了**不存在比 6 枚更少的付法**。

像第二个那样追加条件后，范围就移到 7。`Optimize` 会在条件变化后重新寻找最优值。

用 `Solver` 只能说“在找到的解里最好的那个”，而 `Optimize` 能说“这就是最优”。这是两者最大的区别。

| | `Solver` | `Optimize` |
|---|---|---|
| 目的 | 找出一个满足条件的解 | 找出目标函数最优的解 |
| 添加约束 | `solver.add()` | `opt.add()` |
| 最优化 | 做不到 | `opt.minimize()` / `opt.maximize()` |
| 得到的答案 | 任意一个（不一定最优） | 最优的一个 |
| 最优的保证 | 没有 | 可以用 `lower()` / `upper()` 确认 |

## 写成函数，再换换硬币的种类

同样形状的问题，写成函数就能反复使用。顺便也试试日元。

In [4]:
def min_coin_counts(coin_types, amount):
    """返回用最少枚数凑出金额 amount 的组合"""
    counts = [Int(f"n_{c}") for c in coin_types]
    opt = Optimize()
    for n in counts:
        opt.add(n >= 0)
    opt.add(Sum([c * n for c, n in zip(coin_types, counts)]) == amount)
    opt.minimize(Sum(counts))
    assert opt.check() == sat
    model = opt.model()
    return {c: model[n].as_long() for c, n in zip(coin_types, counts) if model[n].as_long() > 0}


print(min_coin_counts([1, 2, 5, 10, 25, 50], 134))
print(min_coin_counts([1, 5, 10, 50, 100, 500], 1340))

{2: 2, 5: 1, 25: 1, 50: 2}
{10: 4, 100: 3, 500: 2}


只改硬币种类和金额，同一段代码就能用。1340 日元的话，答案是 500 日元×2 + 100 日元×3 + 10 日元×4 共 9 枚，最少。

## 小结

- 约束满足问题是“找满足条件的解”，最优化问题是“在满足条件的解中找最好的”
- `Solver` 能找出一个解（或全部解），但不知道是不是最优
- `Optimize` 用 `minimize()` / `maximize()` 指定目标函数，寻找最优解
- 如果 `lower()` 和 `upper()` 一致，那就是真正的最优值
- 把所有解收集起来的方法，解一多就应付不了（这个问题去掉上限有 11,713 种）